# Experimento: NeuroLKH no TSP

Este notebook demonstra o fluxo completo do NeuroLKH neste projeto: uma SGN treinada com instâncias sintéticas e rótulos ótimos gerados por Held–Karp fornece candidatos e penalidades ao LKH em C, chamado pelo pacote Elkai. A instância de demonstração também é sintética, para que o exemplo seja autocontido e rápido.

> **Limitações:** os rótulos exatos por Held–Karp são limitados a 18 nós. O treino pequeno abaixo demonstra a API, mas não garante generalização para instâncias reais de 250 nós nem qualidade de produção. O Elkai/LKH está sujeito a licença acadêmica e não comercial; consulte a documentação do algoritmo antes de usar ou distribuir.

A implementação reutilizável está em `src/monoobjective_algorithms/tsp/heuristics/neuro_lkh.py`; geração de exemplos e treinamento ficam em `neuro_lkh_training.py`.

In [6]:
!pip install torch

  Using cached torch-2.15.0-cp314-cp314-manylinux_2_28_x86_64.whl.metadata (36 kB)
  Using cached filelock-4.2.7-py3-none-any.whl.metadata (2.0 kB)
  Using cached setuptools-84.2.1-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached cuda_toolkit-13.2.2-py2.py3-none-any.whl.metadata (20 kB)
  Using cached cuda_bindings-13.4.4-cp314-cp314-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached nvidia_cudnn_cu13-9.26.0.51-py3-none-manylinux_2_27_x86_64.whl.metadata (1.9 kB)
  Using cached nvidia_cusparselt_cu13-0.8.1-py3-none-manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached nvidia_nccl_cu13-2.30.7-py3-none-manylinux_2_18_x86_64.whl.metadata (2.1 kB)
  Using cached nvidia_nvshmem_cu13-3.7.2-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.2 kB)
  Using c

In [7]:
from pathlib import Path
import sys
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import torch


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src/monoobjective_algorithms").is_dir():
            return candidate
    raise FileNotFoundError(
        "Não encontrei src/monoobjective_algorithms. Abra o notebook dentro do workspace do projeto."
    )


REPOSITORY_ROOT = find_repository_root(Path.cwd().resolve())
SRC_PATH = str(REPOSITORY_ROOT / "src")
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

from monoobjective_algorithms.interfaces import SolveRequest
from monoobjective_algorithms.tsp.heuristics import NeuroLKH
from monoobjective_algorithms.tsp.heuristics.neuro_lkh_training import (
    generate_synthetic_instance,
    generate_training_examples,
    held_karp_tour,
    save_checkpoint,
    train_model,
)

print(f"Repositório: {REPOSITORY_ROOT}")
print(f"PyTorch: {torch.__version__}")

Repositório: /workspaces/OtimizacaoMultiObjetivo
PyTorch: 2.15.0+cu132


## Treine a SGN

Cada exemplo recebe como rótulo um ciclo ótimo calculado por Held–Karp. Os hiperparâmetros e o número de exemplos foram reduzidos para manter a demonstração leve; para experimentos sérios, aumente o corpus e avalie em instâncias representativas.

In [ ]:
SEED = 42
NODE_COUNT = 8
SAMPLE_COUNT = 8

torch.manual_seed(SEED)
training_examples = generate_training_examples(
    sample_count=SAMPLE_COUNT,
    node_count=NODE_COUNT,
    seed=SEED,
)
model, training_losses = train_model(
    training_examples,
    epochs=2,
    batch_size=4,
    seed=SEED,
    hidden_dim=16,
    gcn_layers=2,
    mlp_layers=2,
)

checkpoint_directory = tempfile.TemporaryDirectory(prefix="neuro-lkh-demo-")
CHECKPOINT_PATH = Path(checkpoint_directory.name) / "neuro_lkh.pt"
save_checkpoint(
    model,
    CHECKPOINT_PATH,
    node_count=NODE_COUNT,
    candidate_count=NODE_COUNT - 1,
)

print(f"Exemplos de treino: {len(training_examples)} ({NODE_COUNT} nós cada)")
print(f"Perda por época: {[round(loss, 4) for loss in training_losses]}")
print(f"Checkpoint temporário: {CHECKPOINT_PATH}")

## Configure e execute o LKH

A matriz de tempo gerada é dirigida, então o wrapper a envia ao backend como ATSP explícito. O LKH faz a busca local por trocas de ordem variável; a SGN orienta os candidatos e as penalidades. Elkai precisa estar instalado conforme `requirements.txt`; se a extensão nativa não estiver disponível, a chamada falha explicitamente.

In [ ]:
rng = np.random.default_rng(SEED + 1)
demo_instance = generate_synthetic_instance(NODE_COUNT, rng)
solver = NeuroLKH(
    model_path=CHECKPOINT_PATH,
    candidate_count=NODE_COUNT - 1,
    candidate_limit=5,
    trials=2,
    seed=SEED,
)
result = solver.solve(demo_instance, SolveRequest(objective="time"))

assert result.status == "feasible"
assert result.solution is not None
route = result.solution.route
assert len(route) == NODE_COUNT + 1
assert route[0] == route[-1] == 1
assert set(route[:-1]) == set(range(1, NODE_COUNT + 1))

optimal_zero_based_tour = held_karp_tour(demo_instance.time_matrix)
optimal_time = sum(
    demo_instance.time_matrix[source][target]
    for source, target in zip(
        optimal_zero_based_tour,
        (*optimal_zero_based_tour[1:], optimal_zero_based_tour[0]),
    )
)

print(f"Status: {result.status}")
print(f"Rota (inclui retorno): {route}")
print(f"Tempo: {result.objective_values['time']:.2f}")
print(f"Distância: {result.objective_values['distance']:.2f}")
print(f"Ótimo exato do tempo (Held–Karp, para comparação): {optimal_time:.2f}")

A solução do LKH é heurística e não tem garantia de ótimo. Held–Karp foi executado aqui apenas para estabelecer um comparativo pequeno; seu custo cresce exponencialmente, por isso o limite de 18 nós.

In [ ]:
objective_names = ["Tempo", "Distância"]
objective_values = [
    result.objective_values["time"],
    result.objective_values["distance"],
]
objective_colors = ["#D96545", "#4777A8"]

fig, (route_ax, objective_ax) = plt.subplots(
    1,
    2,
    figsize=(13, 4.5),
    gridspec_kw={"width_ratios": [1.5, 1]},
    constrained_layout=True,
)

visit_positions = range(len(route))
route_ax.plot(visit_positions, route, color="#28624D", linewidth=1.3)
route_ax.scatter(visit_positions, route, color="#28624D", s=24, zorder=2)
route_ax.set_title("Ordem de visita (inclui retorno à cidade 1)")
route_ax.set_xlabel("Posição na rota")
route_ax.set_ylabel("Identificador da cidade")
route_ax.grid(True, color="#D9D8D0", linewidth=0.6, alpha=0.8)

bars = objective_ax.bar(
    objective_names,
    objective_values,
    color=objective_colors,
    width=0.62,
)
objective_ax.set_title("Custos da rota")
objective_ax.set_ylabel("Valor total")
objective_ax.grid(axis="y", color="#D9D8D0", linewidth=0.6, alpha=0.8)
objective_ax.set_axisbelow(True)
objective_ax.bar_label(bars, fmt="%.2f", padding=4)

fig.suptitle(f"NeuroLKH — ATSP sintético ({NODE_COUNT} cidades)")
plt.show()
plt.close(fig)